# Jina Embeddings Retrieval Evaluation (T4 GPU)

Evaluates 4 Jina embedding models for Arabic WordNet entry retrieval using FAISS.

**Models**: jina-v5-nano (239M), jina-v5-small (677M), jina-v3 (570M), jina-v4 (3.8B)

**Requirements**: T4 GPU runtime (16GB VRAM), ~37MB data upload

## Setup

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-gpu pyyaml

In [ ]:
# Upload and extract the data package
from google.colab import files
import zipfile, os

print("Upload jina_eval_data.zip ...")
uploaded = files.upload()

with zipfile.ZipFile("jina_eval_data.zip", "r") as z:
    z.extractall(".")

os.chdir("retrieval_eval")
!ls -la

In [ ]:
# Quick sanity check
from pathlib import Path
entries = list(Path("export/entries").glob("*.md"))
print(f"Entry files: {len(entries)}")
prepared = list(Path("prepared").iterdir())
print(f"Prepared synsets: {len([p for p in prepared if p.is_dir()])}")

## Run All 4 Jina Models

Each model: download from HuggingFace -> embed 1937 docs -> build FAISS index -> run 126 queries

In [ ]:
# Run jina_v5_nano (239M params, 768 dims) - fastest
!python run_eval.py --backend jina_v5_nano --setup --num-synsets 206 --offset 0 --sleep 0

In [ ]:
# Run jina_v5_small (677M params, 1024 dims)
!python run_eval.py --backend jina_v5_small --setup --num-synsets 206 --offset 0 --sleep 0

In [ ]:
# Run jina_v3 (570M params, 1024 dims)
!python run_eval.py --backend jina_v3 --setup --num-synsets 206 --offset 0 --sleep 0

In [ ]:
# Run jina_v4 (3.8B params, 2048 dims, fp16)
!python run_eval.py --backend jina_v4 --setup --num-synsets 206 --offset 0 --sleep 0

## Analysis

In [ ]:
# Run analysis for all backends
for backend in ["jina_v5_nano", "jina_v5_small", "jina_v3", "jina_v4"]:
    print(f"\n{'='*60}")
    print(f"  {backend}")
    print(f"{'='*60}")
    !python analysis.py --backend {backend}

## Download Results

In [ ]:
# Package all results for download
import zipfile, os
from pathlib import Path

results_zip = "jina_eval_results.zip"
with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for backend in ["jina_v5_nano", "jina_v5_small", "jina_v3", "jina_v4"]:
        run_dir = Path("runs") / backend
        if not run_dir.exists():
            print(f"  Skipping {backend} (no results)")
            continue
        for f in run_dir.iterdir():
            # Skip large FAISS index files - we only need JSON + report
            if f.suffix == ".index":
                continue
            z.write(f, f"runs/{backend}/{f.name}")
            print(f"  Added: runs/{backend}/{f.name}")

print(f"\nResults zip: {results_zip} ({os.path.getsize(results_zip)/1024:.0f} KB)")

from google.colab import files
files.download(results_zip)